In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergedData1211-2.csv"))
dim(data)

[1] 3227892      79

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "num_fac"                                     
 [5] "year"                                        
 [6] "paper_type"                                  
 [7] "paper_language"                              
 [8] "nwbin"                                       
 [9] "new_word_reuse_times"                        
[10] "npbin"                                       
[11] "new_phrase_reuse_times"                      
[12] "novelbin"                                    
[13] "rao_stirling"                                
[14] "author_id"                                   
[15] "author_position"                             
[16] "institution_id"                              
[17] "country_code"                                
[18] "country"                                     
[19] "num_author_log"                              
[20] "num_in

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                     num_fac 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
# data <- data[!is.na(data$Atyp_10pct_Z), ]
# dim(data)
data <- data[data$num_reference >=5, ]
dim(data)

[1] 2768162      79

In [8]:
# data <- data[data$ab_length > 0, ]
# dim(data)
data <- data[data$year >= 1950, ]
dim(data)

[1] 2768162      79

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$year <- as.factor(data$year)

In [11]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log" 
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [12]:
fml <- as.formula(
  paste0("rao_stirling ~ num_fac + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_total <- feols(fml, data = data, vcov = "iid")
summary(model_total)

NOTE: 5 observations removed because of NA values (LHS: 1, RHS: 4).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,157
Fixed-effects: author_id: 78,875,  year: 76
Standard-errors: IID 
                                              Estimate Std. Error    t value
num_fac                                       0.002367   0.000065  36.437746
num_author_log                                0.000054   0.000061   0.888448
num_inst_log                                  0.001220   0.000078  15.664161
num_country_log                              -0.003017   0.000115 -26.343674
num_reference_log                             0.002139   0.000039  54.310433
leadership_global_classallNorth               0.001478   0.000117  12.603633
leadership_global_classallSouth               0.000781   0.000184   4.258577
mean_career_age_log                           0.000564   0.000065   8.692977
inst_h_index_log                              0.000354   0.000050   7.139376
source_h_index_log                           -0.000785   0.000029 -26.938761
core_sourceCore        

In [13]:
pred_curve <- avg_predictions(model_total,
                              variables = list(num_fac = unique(data$num_fac)))
pred_curve

num_fac,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
0,0.1375936,0.0004315549,318.8323,0,Inf,0.1367478,0.1384395
1,0.1399602,0.0004300747,325.4324,0,Inf,0.1391173,0.1408032
2,0.1423268,0.0004383538,324.6848,0,Inf,0.1414677,0.1431860
3,0.1446934,0.0004557879,317.4577,0,Inf,0.1438001,0.1455867
4,0.1470600,0.0004814765,305.4354,0,Inf,0.1461163,0.1480037
5,0.1494266,0.0005140868,290.6641,0,Inf,0.1484190,0.1504341
6,0.1517931,0.0005524837,274.7468,0,Inf,0.1507103,0.1528760
7,0.1541597,0.0005954682,258.8883,0,Inf,0.1529926,0.1553268
8,0.1565263,0.0006421685,243.7465,0,Inf,0.1552677,0.1577849


In [14]:
fname = paste0(main_path, "UseBSForNot/predict_rao_1211_numfac.csv")
write.csv(pred_curve, fname, row.names = FALSE)

In [15]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

disciplines_vars <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [16]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("num_fac", paper_vars, journal_vars),
                           se = "iid",
                           # cluster = ~ paper_year + paper_field,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                 & rao\_stirling\\   
   Model:                              & (1)\\  
   \midrule
   \emph{Variables}\\
   num\_fac                            & 0.002$^{***}$\\   
                                       & ($6.49\times 10^{-5}$)\\    
   num\_author\_log                    & $5.4\times 10^{-5}$\\    
                                       & ($6.08\times 10^{-5}$)\\    
   num\_inst\_log                      & 0.001$^{***}$\\   
                                       & ($7.79\times 10^{-5}$)\\    
   num\_country\_log                   & -0.003$^{***}$\\   
                                       & (0.0001)\\   
   num\_reference\_log                 & 0.002$^{***}$\\   
                                       & ($3.94\times 10^{-5}$)\\    
   leadership\_global\_classallNorth   & 0.001$^{***}$\\   
                                       & (0.0001)\\   
   leadership\_glob

In [17]:
# Without author fixed effects

fml <- as.formula(
  paste0("rao_stirling ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | year")
)

model_total <- feols(fml, data = data, vcov = "iid")
summary(model_total)

NOTE: 5 observations removed because of NA values (LHS: 1, RHS: 4).



OLS estimation, Dep. Var.: rao_stirling
Observations: 2,768,157
Fixed-effects: year: 76
Standard-errors: IID 
                                              Estimate Std. Error    t value
fac_pubBSF                                    0.005240   0.000084  62.265630
num_author_log                               -0.005234   0.000062 -84.237801
num_inst_log                                  0.001767   0.000083  21.413148
num_country_log                               0.000099   0.000124   0.801080
num_reference_log                             0.003217   0.000044  72.491913
leadership_global_classallNorth               0.001766   0.000131  13.518747
leadership_global_classallSouth              -0.002607   0.000146 -17.855436
mean_career_age_log                           0.000148   0.000060   2.463230
inst_h_index_log                              0.001082   0.000042  25.559068
source_h_index_log                           -0.002633   0.000033 -79.194988
core_sourceCore                            

In [18]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("fac_pub", paper_vars, journal_vars),
                           se = "iid",
                           # cluster = ~ paper_year + paper_field,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                 & rao\_stirling\\   
   Model:                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                         & 0.005$^{***}$\\   
                                       & ($8.42\times 10^{-5}$)\\    
   num\_author\_log                    & -0.005$^{***}$\\   
                                       & ($6.21\times 10^{-5}$)\\    
   num\_inst\_log                      & 0.002$^{***}$\\   
                                       & ($8.25\times 10^{-5}$)\\    
   num\_country\_log                   & $9.92\times 10^{-5}$\\    
                                       & (0.0001)\\   
   num\_reference\_log                 & 0.003$^{***}$\\   
                                       & ($4.44\times 10^{-5}$)\\    
   leadership\_global\_classallNorth   & 0.002$^{***}$\\   
                                       & (0.0001)\\   
   leadership\_glo